# Window Functions

## Quick notes
### What this covers
- Window functions vs `GROUP BY`, `OVER (PARTITION BY ... ORDER BY ...)`.
- Ranking: `ROW_NUMBER`, `RANK`, `DENSE_RANK`, `NTILE`, `PERCENT_RANK`, `CUME_DIST`.
- Navigation: `LAG`, `LEAD`, `FIRST_VALUE`, `LAST_VALUE`, `NTH_VALUE`.
- Running totals and moving averages, frames (`ROWS`, `RANGE`, `GROUPS`, `EXCLUDE`), named `WINDOW`.
- Patterns: top-N per group, de-duplication, filtering on window results.
### Key points
- A window function calculates over a set of related rows but keeps every row. `GROUP BY` collapses rows.
- `PARTITION BY` = the groups. `ORDER BY` inside `OVER` = the order within each group. The frame = which rows around the current one are used.
- Window functions run after `WHERE`, `GROUP BY`, `HAVING`, so they only see rows that survived those.
- Any aggregate (`SUM`, `AVG`, `COUNT`, `MIN`, `MAX`) becomes a window function when followed by `OVER`.
### Gotchas (the part worth revising twice)
- You cannot use a window function in `WHERE`. Wrap in a subquery or CTE, then filter.
- With `ORDER BY` in the window, the default frame ends at the current row (and includes ties). So `LAST_VALUE` returns the current row's value, not the last. Fix with an explicit full frame.
- `ROW_NUMBER` breaks ties arbitrarily unless your `ORDER BY` is unique. Add a tiebreaker.
- `RANGE` includes all tied rows, `ROWS` counts physical rows. Running totals differ on ties.
- `LAG`/`LEAD` give NULL at the edges. Pass a default as the third argument.
- A window `ORDER BY` does not sort the final output. Add a normal `ORDER BY` too.
### When it matters
- Rankings, "top 3 per category", running balances, month-over-month change, moving averages, removing duplicates, percentiles.
### One-line recap example
```sql
SELECT name, dept, salary, RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk FROM emp;
```

## 01. Seed data
- Employees (ranking and groups) and daily sales (running totals). Fixed values, with deliberate ties.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE emp (id INTEGER PRIMARY KEY, name TEXT, dept TEXT, salary INTEGER);
INSERT INTO emp VALUES
 (1,'Alice','Eng',9500),(2,'Bob','Eng',7200),(3,'Gus','Eng',7200),(4,'Hana','Eng',6000),
 (5,'Carla','Sales',6800),(6,'Dev','Sales',6100),(7,'Eve','Sales',5800),
 (8,'Finn','Mkt',7500),(9,'Ivy','Mkt',5000);
CREATE TABLE sales (day TEXT PRIMARY KEY, amount INTEGER);
INSERT INTO sales VALUES
 ('2024-01-01',100),('2024-01-02',150),('2024-01-03',150),('2024-01-04',200),
 ('2024-01-05',50),('2024-01-06',300),('2024-01-07',250);
SELECT * FROM emp ORDER BY id;

id | name  | dept  | salary
---+-------+-------+-------
1  | Alice | Eng   | 9500  
2  | Bob   | Eng   | 7200  
3  | Gus   | Eng   | 7200  
4  | Hana  | Eng   | 6000  
5  | Carla | Sales | 6800  
6  | Dev   | Sales | 6100  
7  | Eve   | Sales | 5800  
8  | Finn  | Mkt   | 7500  
9  | Ivy   | Mkt   | 5000  
(9 rows)



## 02. Window vs GROUP BY
- `GROUP BY` returns one row per group. A window function keeps all rows and adds the group number next to each.
- Use a window when you want the detail AND the group figure together.

In [3]:
%%sql
SELECT dept, ROUND(AVG(salary)) AS avg_salary FROM emp GROUP BY dept ORDER BY dept;
SELECT name, dept, salary,
       ROUND(AVG(salary) OVER (PARTITION BY dept)) AS dept_avg,
       salary - ROUND(AVG(salary) OVER (PARTITION BY dept)) AS vs_avg
FROM emp ORDER BY dept, name;

dept  | avg_salary
------+-----------
Eng   | 7475.0    
Mkt   | 6250.0    
Sales | 6233.0    
(3 rows)

name  | dept  | salary | dept_avg | vs_avg 
------+-------+--------+----------+--------
Alice | Eng   | 9500   | 7475.0   | 2025.0 
Bob   | Eng   | 7200   | 7475.0   | -275.0 
Gus   | Eng   | 7200   | 7475.0   | -275.0 
Hana  | Eng   | 6000   | 7475.0   | -1475.0
Finn  | Mkt   | 7500   | 6250.0   | 1250.0 
Ivy   | Mkt   | 5000   | 6250.0   | -1250.0
Carla | Sales | 6800   | 6233.0   | 567.0  
Dev   | Sales | 6100   | 6233.0   | -133.0 
Eve   | Sales | 5800   | 6233.0   | -433.0 
(9 rows)



## 03. OVER (), PARTITION BY, ORDER BY
- `OVER ()` = the whole result is one window.
- `PARTITION BY dept` = a separate window per department.
- `ORDER BY` inside `OVER` = order inside each window (and it changes the default frame, see section 12).

In [4]:
%%sql
SELECT name, dept, salary,
       SUM(salary) OVER () AS company_total,
       SUM(salary) OVER (PARTITION BY dept) AS dept_total,
       ROUND(100.0 * salary / SUM(salary) OVER (PARTITION BY dept), 1) AS pct_of_dept
FROM emp ORDER BY dept, name;

name  | dept  | salary | company_total | dept_total | pct_of_dept
------+-------+--------+---------------+------------+------------
Alice | Eng   | 9500   | 61100         | 29900      | 31.8       
Bob   | Eng   | 7200   | 61100         | 29900      | 24.1       
Gus   | Eng   | 7200   | 61100         | 29900      | 24.1       
Hana  | Eng   | 6000   | 61100         | 29900      | 20.1       
Finn  | Mkt   | 7500   | 61100         | 12500      | 60.0       
Ivy   | Mkt   | 5000   | 61100         | 12500      | 40.0       
Carla | Sales | 6800   | 61100         | 18700      | 36.4       
Dev   | Sales | 6100   | 61100         | 18700      | 32.6       
Eve   | Sales | 5800   | 61100         | 18700      | 31.0       
(9 rows)



## 04. ROW_NUMBER, RANK, DENSE_RANK
- `ROW_NUMBER`: 1,2,3,4 always unique, ties broken arbitrarily.
- `RANK`: ties share a rank, then a gap (1,2,2,4).
- `DENSE_RANK`: ties share a rank, no gap (1,2,2,3).
- Bob and Gus tie at 7200 in Eng.

In [5]:
%%sql
SELECT name, salary,
       ROW_NUMBER() OVER w AS row_num,
       RANK()       OVER w AS rnk,
       DENSE_RANK() OVER w AS dense_rnk
FROM emp WHERE dept = 'Eng'
WINDOW w AS (ORDER BY salary DESC)
ORDER BY salary DESC, name;

name  | salary | row_num | rnk | dense_rnk
------+--------+---------+-----+----------
Alice | 9500   | 1       | 1   | 1        
Bob   | 7200   | 2       | 2   | 2        
Gus   | 7200   | 3       | 2   | 2        
Hana  | 6000   | 4       | 4   | 3        
(4 rows)



## 05. Make ROW_NUMBER deterministic
- If the `ORDER BY` has ties, `ROW_NUMBER` may order the tied rows differently from run to run or version to version.
- Add a unique tiebreaker such as the primary key.

In [6]:
%%sql
SELECT name, salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC, id) AS stable_num
FROM emp WHERE dept = 'Eng' ORDER BY stable_num;

name  | salary | stable_num
------+--------+-----------
Alice | 9500   | 1         
Bob   | 7200   | 2         
Gus   | 7200   | 3         
Hana  | 6000   | 4         
(4 rows)



## 06. Top-N per group
- Rank inside each group, then keep rank <= N.
- You cannot filter on the window function in the same query's `WHERE`, so use a CTE.
- Bad vs good shown below.

In [7]:
%%sql expect_error
-- Bad: window functions are not allowed in WHERE
SELECT name FROM emp WHERE ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) <= 2;

ERROR: misuse of window function ROW_NUMBER()



In [8]:
%%sql
-- Good: top 2 per department
WITH ranked AS (
    SELECT name, dept, salary,
           ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC, id) AS rn
    FROM emp
)
SELECT dept, name, salary, rn FROM ranked WHERE rn <= 2 ORDER BY dept, rn;

dept  | name  | salary | rn
------+-------+--------+---
Eng   | Alice | 9500   | 1 
Eng   | Bob   | 7200   | 2 
Mkt   | Finn  | 7500   | 1 
Mkt   | Ivy   | 5000   | 2 
Sales | Carla | 6800   | 1 
Sales | Dev   | 6100   | 2 
(6 rows)



## 07. Top-N with ties (RANK vs ROW_NUMBER)
- "Top 2 salaries" with `RANK` keeps everyone tied for the position. `ROW_NUMBER` cuts the tie arbitrarily.
- Pick the one that matches the business question.

In [9]:
%%sql
WITH r AS (
    SELECT name, salary, RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM emp WHERE dept = 'Eng'
)
SELECT name, salary, rnk FROM r WHERE rnk <= 2 ORDER BY rnk, name;

name  | salary | rnk
------+--------+----
Alice | 9500   | 1  
Bob   | 7200   | 2  
Gus   | 7200   | 2  
(3 rows)



## 08. De-duplicate with ROW_NUMBER
- Number the rows inside each set of duplicates, keep number 1, and delete the rest.
- Pick the "keeper" with the `ORDER BY` (here the earliest sign-up).

In [10]:
%%sql
CREATE TABLE signups (id INTEGER PRIMARY KEY, email TEXT, created TEXT);
INSERT INTO signups VALUES
 (1,'a@x.com','2024-01-01'),(2,'b@x.com','2024-01-02'),(3,'a@x.com','2024-01-03'),(4,'a@x.com','2024-01-04');
DELETE FROM signups WHERE id IN (
    SELECT id FROM (
        SELECT id, ROW_NUMBER() OVER (PARTITION BY email ORDER BY created, id) AS rn FROM signups
    ) WHERE rn > 1
);
SELECT * FROM signups ORDER BY id;

id | email   | created   
---+---------+-----------
1  | a@x.com | 2024-01-01
2  | b@x.com | 2024-01-02
(2 rows)



## 09. NTILE
- Splits the ordered rows into N buckets of near-equal size (quartiles, deciles).
- When rows do not divide evenly, the first buckets get one extra row.

In [11]:
%%sql
SELECT name, salary, NTILE(3) OVER (ORDER BY salary DESC, id) AS bucket
FROM emp ORDER BY salary DESC, id;

name  | salary | bucket
------+--------+-------
Alice | 9500   | 1     
Finn  | 7500   | 1     
Bob   | 7200   | 1     
Gus   | 7200   | 2     
Carla | 6800   | 2     
Dev   | 6100   | 2     
Hana  | 6000   | 3     
Eve   | 5800   | 3     
Ivy   | 5000   | 3     
(9 rows)



## 10. LAG and LEAD
- `LAG(col, n, default)` = value n rows before. `LEAD` = n rows after.
- First/last rows have no neighbour, so you get NULL unless you pass a default.
- Great for day-over-day or month-over-month change.

In [12]:
%%sql
SELECT day, amount,
       LAG(amount)       OVER (ORDER BY day) AS prev_null_at_start,
       LAG(amount, 1, 0) OVER (ORDER BY day) AS prev_default_0,
       amount - LAG(amount, 1, amount) OVER (ORDER BY day) AS change,
       LEAD(amount, 2)   OVER (ORDER BY day) AS two_days_ahead
FROM sales ORDER BY day;

day        | amount | prev_null_at_start | prev_default_0 | change | two_days_ahead
-----------+--------+--------------------+----------------+--------+---------------
2024-01-01 | 100    | NULL               | 0              | 0      | 150           
2024-01-02 | 150    | 100                | 100            | 50     | 200           
2024-01-03 | 150    | 150                | 150            | 0      | 50            
2024-01-04 | 200    | 150                | 150            | 50     | 300           
2024-01-05 | 50     | 200                | 200            | -150   | 250           
2024-01-06 | 300    | 50                 | 50             | 250    | NULL          
2024-01-07 | 250    | 300                | 300            | -50    | NULL          
(7 rows)



## 11. FIRST_VALUE, NTH_VALUE
- `FIRST_VALUE(col)`: value from the first row of the frame. `NTH_VALUE(col, n)`: the n-th row of the frame (NULL if the frame is shorter).
- Compare every salary to the top salary in its department.

In [13]:
%%sql
SELECT name, dept, salary,
       FIRST_VALUE(name)   OVER w AS top_earner,
       salary - FIRST_VALUE(salary) OVER w AS gap_to_top,
       NTH_VALUE(name, 2)  OVER w AS second_so_far
FROM emp
WINDOW w AS (PARTITION BY dept ORDER BY salary DESC, id)
ORDER BY dept, salary DESC, id;

name  | dept  | salary | top_earner | gap_to_top | second_so_far
------+-------+--------+------------+------------+--------------
Alice | Eng   | 9500   | Alice      | 0          | NULL         
Bob   | Eng   | 7200   | Alice      | -2300      | Bob          
Gus   | Eng   | 7200   | Alice      | -2300      | Bob          
Hana  | Eng   | 6000   | Alice      | -3500      | Bob          
Finn  | Mkt   | 7500   | Finn       | 0          | NULL         
Ivy   | Mkt   | 5000   | Finn       | -2500      | Ivy          
Carla | Sales | 6800   | Carla      | 0          | NULL         
Dev   | Sales | 6100   | Carla      | -700       | Dev          
Eve   | Sales | 5800   | Carla      | -1000      | Dev          
(9 rows)



## 12. LAST_VALUE and the default-frame gotcha
- With an `ORDER BY`, the default frame is "start of partition up to the current row". So `LAST_VALUE` is just the current row.
- To see the real last row, widen the frame: `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.
- The same default hits `NTH_VALUE` (above, `second_so_far` is NULL on each department's first row).

In [14]:
%%sql
SELECT name, dept, salary,
       LAST_VALUE(name) OVER (PARTITION BY dept ORDER BY salary DESC, id) AS wrong_last,
       LAST_VALUE(name) OVER (PARTITION BY dept ORDER BY salary DESC, id
                              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS lowest_paid
FROM emp ORDER BY dept, salary DESC, id;

name  | dept  | salary | wrong_last | lowest_paid
------+-------+--------+------------+------------
Alice | Eng   | 9500   | Alice      | Hana       
Bob   | Eng   | 7200   | Bob        | Hana       
Gus   | Eng   | 7200   | Gus        | Hana       
Hana  | Eng   | 6000   | Hana       | Hana       
Finn  | Mkt   | 7500   | Finn       | Ivy        
Ivy   | Mkt   | 5000   | Ivy        | Ivy        
Carla | Sales | 6800   | Carla      | Eve        
Dev   | Sales | 6100   | Dev        | Eve        
Eve   | Sales | 5800   | Eve        | Eve        
(9 rows)



## 13. Running totals
- `SUM(...) OVER (ORDER BY ...)` gives a running total, because the default frame is start-to-current.
- Write the frame explicitly so the intention is clear.

In [15]:
%%sql
SELECT day, amount,
       SUM(amount) OVER (ORDER BY day ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
       COUNT(*)    OVER (ORDER BY day) AS day_number
FROM sales ORDER BY day;

day        | amount | running_total | day_number
-----------+--------+---------------+-----------
2024-01-01 | 100    | 100           | 1         
2024-01-02 | 150    | 250           | 2         
2024-01-03 | 150    | 400           | 3         
2024-01-04 | 200    | 600           | 4         
2024-01-05 | 50     | 650           | 5         
2024-01-06 | 300    | 950           | 6         
2024-01-07 | 250    | 1200          | 7         
(7 rows)



## 14. Moving average (ROWS frame)
- `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` = this row and the two before it.
- The first rows average over fewer rows (the frame is cut off at the start). Count the frame size if that matters.

In [16]:
%%sql
SELECT day, amount,
       ROUND(AVG(amount) OVER (ORDER BY day ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS avg_3day,
       COUNT(amount)     OVER (ORDER BY day ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rows_in_frame,
       ROUND(AVG(amount) OVER (ORDER BY day ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING), 1) AS centered_avg
FROM sales ORDER BY day;

day        | amount | avg_3day | rows_in_frame | centered_avg
-----------+--------+----------+---------------+-------------
2024-01-01 | 100    | 100.0    | 1             | 125.0       
2024-01-02 | 150    | 125.0    | 2             | 133.3       
2024-01-03 | 150    | 133.3    | 3             | 166.7       
2024-01-04 | 200    | 166.7    | 3             | 133.3       
2024-01-05 | 50     | 133.3    | 3             | 183.3       
2024-01-06 | 300    | 183.3    | 3             | 200.0       
2024-01-07 | 250    | 200.0    | 3             | 275.0       
(7 rows)



## 15. ROWS vs RANGE vs GROUPS
- `ROWS`: counts physical rows. `RANGE`: by value of the `ORDER BY` column, so tied rows (peers) all come along. `GROUPS`: counts whole groups of tied rows.
- Days 2 and 3 both have 150, so they are peers.
- Look at `range_total` on day 2: it already includes day 3. `rows_total` does not.

In [17]:
%%sql
SELECT day, amount,
       SUM(amount) OVER (ORDER BY amount ROWS   BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rows_total,
       SUM(amount) OVER (ORDER BY amount RANGE  BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS range_total,
       SUM(amount) OVER (ORDER BY amount GROUPS BETWEEN 1 PRECEDING AND CURRENT ROW)         AS groups_total
FROM sales ORDER BY amount, day;

day        | amount | rows_total | range_total | groups_total
-----------+--------+------------+-------------+-------------
2024-01-05 | 50     | 50         | 50          | 50          
2024-01-01 | 100    | 150        | 150         | 150         
2024-01-02 | 150    | 300        | 450         | 400         
2024-01-03 | 150    | 450        | 450         | 400         
2024-01-04 | 200    | 650        | 650         | 500         
2024-01-07 | 250    | 900        | 900         | 450         
2024-01-06 | 300    | 1200       | 1200        | 550         
(7 rows)



## 16. RANGE with offsets and EXCLUDE
- With numeric `ORDER BY`, `RANGE BETWEEN 50 PRECEDING AND 50 FOLLOWING` means "values within 50 of mine".
- `EXCLUDE CURRENT ROW` leaves out the row itself (others only). Also: `EXCLUDE GROUP`, `EXCLUDE TIES`, `EXCLUDE NO OTHERS` (default).

In [18]:
%%sql
SELECT day, amount,
       COUNT(*) OVER (ORDER BY amount RANGE BETWEEN 50 PRECEDING AND 50 FOLLOWING) AS within_50,
       SUM(amount) OVER (ORDER BY day ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING EXCLUDE CURRENT ROW) AS neighbours_sum,
       SUM(amount) OVER (ORDER BY amount RANGE BETWEEN CURRENT ROW AND CURRENT ROW EXCLUDE CURRENT ROW) AS tied_others_sum
FROM sales ORDER BY day;

day        | amount | within_50 | neighbours_sum | tied_others_sum
-----------+--------+-----------+----------------+----------------
2024-01-01 | 100    | 4         | 150            | NULL           
2024-01-02 | 150    | 4         | 250            | 150            
2024-01-03 | 150    | 4         | 350            | 150            
2024-01-04 | 200    | 4         | 200            | NULL           
2024-01-05 | 50     | 2         | 500            | NULL           
2024-01-06 | 300    | 2         | 300            | NULL           
2024-01-07 | 250    | 3         | 300            | NULL           
(7 rows)



## 17. PERCENT_RANK and CUME_DIST
- `PERCENT_RANK` = (rank - 1) / (rows - 1): 0 for the lowest, 1 for the highest.
- `CUME_DIST` = share of rows with a value less than or equal to this one (never 0).

In [19]:
%%sql
SELECT name, salary,
       RANK()         OVER w AS rnk,
       ROUND(PERCENT_RANK() OVER w, 2) AS pct_rank,
       ROUND(CUME_DIST()    OVER w, 2) AS cume_dist
FROM emp WHERE dept = 'Eng'
WINDOW w AS (ORDER BY salary)
ORDER BY salary, name;

name  | salary | rnk | pct_rank | cume_dist
------+--------+-----+----------+----------
Hana  | 6000   | 1   | 0.0      | 0.25     
Bob   | 7200   | 2   | 0.33     | 0.75     
Gus   | 7200   | 2   | 0.33     | 0.75     
Alice | 9500   | 4   | 1.0      | 1.0      
(4 rows)



## 18. Aggregate window functions side by side
- `SUM`, `AVG`, `MIN`, `MAX`, `COUNT` with `OVER (PARTITION BY ...)` put group facts on every row.
- Use them for "share of group", "difference from group max", "rows in group".

In [20]:
%%sql
SELECT name, dept, salary,
       COUNT(*)   OVER p AS dept_size,
       MIN(salary) OVER p AS dept_min,
       MAX(salary) OVER p AS dept_max,
       ROUND(100.0 * (salary - MIN(salary) OVER p) / (MAX(salary) OVER p - MIN(salary) OVER p)) AS pct_in_range
FROM emp
WINDOW p AS (PARTITION BY dept)
ORDER BY dept, name;

name  | dept  | salary | dept_size | dept_min | dept_max | pct_in_range
------+-------+--------+-----------+----------+----------+-------------
Alice | Eng   | 9500   | 4         | 6000     | 9500     | 100.0       
Bob   | Eng   | 7200   | 4         | 6000     | 9500     | 34.0        
Gus   | Eng   | 7200   | 4         | 6000     | 9500     | 34.0        
Hana  | Eng   | 6000   | 4         | 6000     | 9500     | 0.0         
Finn  | Mkt   | 7500   | 2         | 5000     | 7500     | 100.0       
Ivy   | Mkt   | 5000   | 2         | 5000     | 7500     | 0.0         
Carla | Sales | 6800   | 3         | 5800     | 6800     | 100.0       
Dev   | Sales | 6100   | 3         | 5800     | 6800     | 30.0        
Eve   | Sales | 5800   | 3         | 5800     | 6800     | 0.0         
(9 rows)



## 19. Named WINDOW clause
- Define a window once with `WINDOW w AS (...)` and reuse it by name. Less repetition, fewer typos.
- A named window can be extended: `OVER (w ROWS BETWEEN ...)` adds a frame to the base definition.
- `WINDOW` goes after `HAVING` and before `ORDER BY`.

In [21]:
%%sql
SELECT day, amount,
       SUM(amount) OVER w AS running,
       AVG(amount) OVER (w ROWS BETWEEN 1 PRECEDING AND CURRENT ROW) AS avg_2day
FROM sales
WINDOW w AS (ORDER BY day)
ORDER BY day;

day        | amount | running | avg_2day
-----------+--------+---------+---------
2024-01-01 | 100    | 100     | 100.0   
2024-01-02 | 150    | 250     | 125.0   
2024-01-03 | 150    | 400     | 150.0   
2024-01-04 | 200    | 600     | 175.0   
2024-01-05 | 50     | 650     | 125.0   
2024-01-06 | 300    | 950     | 175.0   
2024-01-07 | 250    | 1200    | 275.0   
(7 rows)



## 20. Filtering on window results
- `WHERE` runs before windows. To filter on a window value, compute it in a subquery or CTE, then filter outside.
- Example: employees paid above their department average.

In [22]:
%%sql
WITH x AS (
    SELECT name, dept, salary, AVG(salary) OVER (PARTITION BY dept) AS dept_avg FROM emp
)
SELECT name, dept, salary, ROUND(dept_avg) AS dept_avg FROM x WHERE salary > dept_avg ORDER BY dept, name;

name  | dept  | salary | dept_avg
------+-------+--------+---------
Alice | Eng   | 9500   | 7475.0  
Finn  | Mkt   | 7500   | 6250.0  
Carla | Sales | 6800   | 6233.0  
(3 rows)



## 21. Windows over grouped results
- Order of work: `WHERE`, `GROUP BY`, `HAVING`, then windows. So you can window over the grouped rows.
- Example: rank departments by total pay.

In [23]:
%%sql
SELECT dept, SUM(salary) AS total,
       RANK() OVER (ORDER BY SUM(salary) DESC) AS dept_rank
FROM emp GROUP BY dept ORDER BY dept_rank;

dept  | total | dept_rank
------+-------+----------
Eng   | 29900 | 1        
Sales | 18700 | 2        
Mkt   | 12500 | 3        
(3 rows)



## 22. Practical: gaps in a sequence and month-over-month
- `LAG` finds changes and gaps. Here: flag days where sales fell compared with the day before.
- Related grouping and joining topics: 02-Querying/03-Subqueries-And-CTEs and 02-Querying/01-Joins.

In [24]:
%%sql
WITH d AS (
    SELECT day, amount, LAG(amount) OVER (ORDER BY day) AS prev FROM sales
)
SELECT day, amount, prev, amount - prev AS change,
       CASE WHEN amount < prev THEN 'DOWN' WHEN amount > prev THEN 'up' ELSE 'flat' END AS trend
FROM d WHERE prev IS NOT NULL ORDER BY day;

day        | amount | prev | change | trend
-----------+--------+------+--------+------
2024-01-02 | 150    | 100  | 50     | up   
2024-01-03 | 150    | 150  | 0      | flat 
2024-01-04 | 200    | 150  | 50     | up   
2024-01-05 | 50     | 200  | -150   | DOWN 
2024-01-06 | 300    | 50   | 250    | up   
2024-01-07 | 250    | 300  | -50    | DOWN 
(6 rows)

